In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import random
from mlp import ModelConfig
from mlp import MLP

%load_ext autoreload
%autoreload 2

words = open("../names.txt", "r").read().splitlines()
torch.set_printoptions(sci_mode=False, precision=4)

torch.manual_seed(42)
random.seed(42)

In [2]:
# 1. create vocab table (a-z + '.')
chars = sorted(list(set("".join(words))))
stoi = {s: i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}
vocab_size = len(itos)

# 2. Data set Train (80%), Dev (10%), Test (10%)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))
words_train = words[: n1]
words_dev = words[n1: n2]
words_test = words[n2: ]

print(f"Total words: {len(words)}")
print(f"Train: {len(words_train)}, Dev: {len(words_dev)}, Test: {len(words_test)}")

Total words: 32033
Train: 25626, Dev: 3203, Test: 3204


In [3]:
#3. Randomly select batch_size from words_list，build idx 和 targets in (b, t)
def get_batch(words_list, batch_size=32, max_len=16):
    ix = torch.randint(0, len(words_list), (batch_size,))
    selected_words = [words_list[i] for i in ix]

    idx = []
    target = []
    for w in selected_words:
        # map each char to ID，and append '.' (0)
        chs = [stoi[c] for c in w] + [stoi['.']]

        # input x:   'e', 'm', 'm', 'a', '.'
        # targe y:   'm', 'm', 'a', '.', '.'
        x = chs
        y = chs[1:] + [stoi['.']]

        # add padding so each idx/target is max_len
        pad_len = max_len - len(x)
        if pad_len > 0:
            # idx Padding use <BLANK> (index 27)
            x += [vocab_size] * pad_len
            # targets Padding use -1 (will be ignored by ignore_index)
            y += [-1] * pad_len
        else:
            x = x[: max_len]
            y = y[: max_len]
        idx.append(x)
        target.append(y)

    return torch.tensor(idx, dtype=torch.long), torch.tensor(target, dtype=torch.long)


In [4]:
#4. Model initialization
config = ModelConfig(block_size=3, vocab_size=vocab_size, n_embed=10, n_embed2=100)
model = MLP(config)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model Total Parameters: {total_params}")

Model Total Parameters: 6107


In [5]:
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01, weight_decay=0.0001)
loss_history = []
max_steps = 20000
batch_size = 32

print("Starting Training...")
for step in range(max_steps):
    #get sampled mini-batch from training data set
    idx, target = get_batch(words_train, batch_size, 16)

    #forward
    logits, loss = model(idx, target)

    #backward
    optimizer.zero_grad()
    loss.backward()

    #learning rate decay
    lr = 0.01 if step < 10000 else 0.001
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    #update params
    optimizer.step()

    loss_history.append(loss.item())

    if step % 5000 == 0 or step == max_steps - 1:
        print(f"Step {step:5d}/{max_steps} | Loss: {loss.item():.4f} | LR: {lr}")


Starting Training...
Step     0/20000 | Loss: 3.3025 | LR: 0.01
Step  5000/20000 | Loss: 1.8302 | LR: 0.01
Step 10000/20000 | Loss: 1.6909 | LR: 0.001
Step 15000/20000 | Loss: 1.6739 | LR: 0.001
Step 19999/20000 | Loss: 1.6021 | LR: 0.001
